In [ ]:
# Train custom CellTypist models and cross-annotate sc and sn.
!pip install celltypist

In [ ]:
import os
import pandas as pd
import scanpy as sc
import glob
import anndata as ad
import celltypist as ct

In [ ]:
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    frameon=True,  
    format="pdf"   
)

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/atlas_project/heart_sc_old/qc3_scvi.h5ad')
adata1 = sc.read('/home/yanjun3/atlas/heart_sn/sn_final.h5ad')

In [ ]:
common = adata.var_names.intersection(adata1.var_names)
adata  = adata[:, common].copy()
adata1 = adata1[:, common].copy()

In [ ]:
model_fs = ct.train(adata, labels = 'new_anno', n_jobs = 10, feature_selection = True)

In [ ]:
adata1.obs.rename(columns={"new_anno": "new_anno_sn"}, inplace=True)

In [ ]:
model_fs.write("heartsc_custom.pkl")

In [ ]:
#model_loaded = ct.models.Model.load("/home/yanjun3/owen/GSE155960/WAT_custom.pkl")

In [ ]:
predictions = ct.annotate(adata1, model = model_fs, majority_voting = True)

In [ ]:
adata1 = predictions.to_adata()

In [ ]:
#sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata1, color = ['new_anno_sn', 'predicted_labels', 'majority_voting'], legend_loc = 'right margin', ncols = 1)

In [ ]:
ct.dotplot(predictions, use_as_reference = 'new_anno_sn', use_as_prediction = 'majority_voting')

In [ ]:
adata1.obs.rename(columns={"new_anno_sn": "new_anno"}, inplace=True)
adata.obs.rename(columns={"new_anno": "new_anno_sc"}, inplace=True)

In [ ]:
model_fs = ct.train(adata1, labels = 'new_anno', n_jobs = 10, feature_selection = True)

In [ ]:
model_fs.write("heartsn_custom.pkl")

In [ ]:
predictions = ct.annotate(adata, model = model_fs, majority_voting = True)

In [ ]:
adata = predictions.to_adata()

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color = ['new_anno_sc', 'predicted_labels', 'majority_voting'], legend_loc = 'right margin', ncols = 1)